# 02. Térbeli mérések

## MIT mérünk?

Ez a fejezet a lakások **térbeli helyzetét** méri három rétegben: (1) a vasúti vágánytól mért légvonali távolságból képzett **immissziós sávok** (a zaj- és rezgésterhelés kategóriái), (2) az úthálózaton mért gyalogos **izokrónok** (hány perc séta a legközelebbi állomás/megálló: 5-10-15 perces gyűrűk), és (3) a gyalogosan elérhető szolgáltatások (**POI-k**) száma ugyanezen a 15 perces körzeten belül. Emellett az árak **térbeli eloszlását** ábrázoljuk: ponttérkép, interpolált árfelület (IDW) és háromdimenziós értékdomborzat (2D csúcs–völgy annotációval, 3D felülettel, metrikaválasztóval).

## MIÉRT fontos?

Egy lakás árát a belső jellemzők mellett az határozza meg, *mi érhető el tőle gyalogosan*, és *mi terheli a közvetlen környezetében*. A két hatás iránya ellentétes lehet: ugyanaz a vasúti infrastruktúra egyrészt elérhetőségi előny (gyors eljutás a városközpontba), másrészt környezeti teher (zaj, rezgés) — a kettőt ezért külön-külön, eltérő mérési logikával kell megfogni, mielőtt egy modellben együtt szerepeltetnénk őket.

## HOGYAN mérünk? Küszöbök kontra folytonos távolság

A mérés két elvet kever. A **környezeti terhelést sávosan** mérjük: a vágánytól mért légvonali távolságot rögzített küszöbökkel (0–150–300–500–1000–2000 méter) osztjuk sávokra, mert a zaj és a rezgés a forrástól távolodva erősen, nem egyenletesen csillapodik — a küszöbök a csillapodási tartományokhoz igazodnak. A **gyalogos elérhetőséget szintén küszöbösen** mérjük, de hálózaton: 1,25 m/s sétatempóval a 375 méter 5 perc, a 750 méter 10 perc, az 1125 méter 15 perc sétatáv — ezek az izokrón-sávok, mert a mindennapi döntések („elérhető-e gyalog?") valójában küszöbdöntések. Emellett mindkét dimenzióban rendelkezésre áll a **folytonos** mérték is (méterben), mert a későbbi modellek a folytonos alakot használják.

## Mi a referencia elve és a döntési szabály?

Minden sávhatást egy **referenciához** viszonyítunk: az a legtávolabbi, még kellően népes (legalább 20 megfigyelésű) sáv, amely a terheléstől gyakorlatilag függetlennek tekinthető — az izokrónoknál ez a 1125 méteren kívüli zóna. A sávok relatív helyzetét így a referencia-kontraszt adja. A fejezet minden számát a közös elemző modul térbeli méréseiből (gyorsítótárból) veszi: a notebook nem számol újra, csak megjelenít.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os; sys.path.insert(0, '..')
from ingatlan_tdk._utils import *
from ingatlan_tdk import analyses as A
setup_plotly()
AREA = os.environ.get('TDK_ACTIVE_AREA', 'kobanya')
df = load_szamitott_master(); p = A.pontos_elado()


In [ ]:
import plotly.express as px
# A terület neve és fejléce dinamikusan a konfigurációból
from IPython.display import display, HTML, Markdown

area_name = get_area_metadata(os.environ.get('TDK_ACTIVE_AREA', 'kobanya'))['name']
display(HTML(render_area_header_html(area=AREA, df=df)))
print(f"Aktív terület: {area_name}  |  teljes minta: {len(df):,} sor  |  elemzési minta (pontos, eladó): {len(p):,} sor".replace(',', ' '))


## 1. Immissziós sávok és gyalogos izokrónok — a mérési táblák

**MIT látunk?** Két táblázatot. Az immissziós tábla a rögzített 0-150-300-500-1000-2000 méteres sávok elemszámait mutatja (hány lakás esik az egyes vágánytávolság-sávokba). Az izokrón-tábla hálózatonként (vasút, metró, villamos, kötöttpálya-minimum) azt, hogy hány lakásból érhető el az adott megálló 5, 10, illetve 15 perc sétával, továbbá a folytonos hálózati távolságok kitöltöttségét és mediánját.

**MIÉRT ezek a küszöbök?** Az immissziós küszöbök a zaj- és rezgéscsillapodás tartományait követik: 150 méteren belül közvetlen, 150–300 méter között erős, 300–500 méter között mérsékelt, 500–1000 és 1000–2000 méter között gyengülő, 2000 méteren túl elhanyagolható terheléssel számolunk. Az izokrón-küszöbök a sétaidőből adódnak: 75 méter/perc (1,25 m/s) mellett a 375/750/1125 méter éppen az 5/10/15 perces gyűrű.

**HOGYAN?** A sávok képzése a feldolgozó láncban, a gyorsítótárban történt; a notebook csak megjeleníti az eredményeket, oszlopdiagramokkal kiegészítve.

**Döntési szabály:** a sávhatások értelmezése mindig a referencia-kontraszt alapján történik (a legtávolabbi, legalább 20 elemű sáv, illetve a 1125 méteren kívüli zóna a viszonyítási alap); a folytonos méter-távolság a későbbi modellek bemenete.


In [ ]:
# 1. Térbeli mérések a cache-ből (A.spatial_metrics) + KPI-k
S = A.spatial_metrics()

vasut15 = int(S['izokronok'].loc[S['izokronok']['halozat'] == 'vasut', '15p_N'].iloc[0])
kp15 = int(S['izokronok'].loc[S['izokronok']['halozat'] == 'kotottpalya', '15p_N'].iloc[0])
poi15_mean = float(S['poi'].loc[S['poi']['poi'] == 'poi_15p_count', 'atlag'].iloc[0])
sav1 = int(S['immisszios_savok']['N'].iloc[0])
med_nm = float(p['nm_ar_huf'].median())
n_p = len(p)

kpi2 = [
    ("Elemzési minta", f"{n_p} db", "Pontos koordinátás eladó lakások", "#1e3a8a"),
    ("Vasút 15p séta", f"{vasut15} db", f"{vasut15/n_p*100:.1f}% lefedettség", "#ec4899"),
    ("Kötöttpálya 15p séta", f"{kp15} db", f"{kp15/n_p*100:.1f}% lefedettség", "#2563eb"),
    ("Átlagos POI 15p", f"{poi15_mean:.1f} db", "Hálózati 15-perces ellátottság", "#059669"),
    ("<150 m immisszió", f"{sav1} db", "Közvetlen vágányközeli sáv", "#dc2626"),
    ("Medián ár/m² (pontos)", fmt_huf(med_nm), "Elemzési minta", "#7c3aed"),
]
display(HTML(kpi_grid_html(kpi2)))

iso = S['izokronok'].copy()
iso.columns = ['Hálózat', '5p (N)', '10p (N)', '15p (N)', 'Folytonos kitöltött', 'Medián táv (m)']
display(HTML("<b>Gyalogos izokrónok (0-375 / 375-750 / 750-1125 m, 1,25 m/s):</b><br>"
             + iso.to_html(classes='table table-bordered table-striped', index=False)))

sav = S['immisszios_savok'].copy()
sav.columns = ['Immissziós sáv', 'N']
display(HTML("<b>Immissziós sávok (légvonali vágánytávolság, 0-150-300-500-1000-2000 m):</b><br>"
             + sav.to_html(classes='table table-bordered table-striped', index=False)))

# Oszlopdiagramok
iso_long = S['izokronok'].melt(id_vars=['halozat'], value_vars=['5p_N', '10p_N', '15p_N'],
                              var_name='izokrón', value_name='N')
iso_long['izokrón'] = iso_long['izokrón'].map({'5p_N': '0-375 m (5 perc)',
                                               '10p_N': '375-750 m (10 perc)',
                                               '15p_N': '750-1125 m (15 perc)'})
fig_iso = px.bar(iso_long, x='halozat', y='N', color='izokrón', barmode='group',
                 title='Gyalogos elérhetőség hálózatonként (lakásszám a sávokban)',
                 labels={'halozat': 'Hálózat', 'N': 'Lakások száma'}, template=PLOTLY_TEMPLATE, height=420)
fig_iso.show()

fig_sav = px.bar(sav, x='Immissziós sáv', y='N', color='N', color_continuous_scale='OrRd', text='N',
                 title='Immissziós sávok elemszámai (elemzési minta)',
                 labels={'N': 'Lakások száma'}, template=PLOTLY_TEMPLATE, height=400)
fig_sav.update_layout(coloraxis_showscale=False)
fig_sav.show()

print(f"Az elemzési minta {n_p} lakásából {vasut15} ({vasut15/n_p*100:.1f}%) éri el a legközelebbi vasútállomást "
      f"15 perc sétán belül; a közvetlen immissziós sávba (<150 m) {sav1} lakás esik.")


## 2. Szolgáltatási ellátottság (POI) a 15 perces körzetben

**MIT mérünk?** Hány darab szolgáltatás (üzlet, rendelő, gyógyszertár, iskola, óvoda, közlekedési pont stb.) érhető el gyalogosan 375, 750, illetve 1125 méteres hálózati sávon belül.

**MIÉRT?** A „15 perces város" gondolata szerint az életminőséget és a lokáció értékét az határozza meg, hogy a napi szükségletek sétatávon belül elérhetők-e. A POI-szám az elérhetőségi előny közvetlen, sávos mérőszáma.

**HOGYAN?** A POI-pontok nyílt térképi (OpenStreetMap-alapú) adatbázisból származnak; a sávonkénti darabszámok a hálózati elérhetőség alapján készültek a feldolgozó láncban, és a gyorsítótárban tárolódnak. A fejezet a sávonkénti átlagokat és a térképi eloszlást mutatja.

**Döntési szabály:** a POI-sávok kumulatívak (a 15 perces szám tartalmazza a közelebbieket is); az értelmezés az átlagos ellátottság és a területen belüli egyenlőtlenség együttes leírása.


In [ ]:
# 2. POI-statisztika a cache-ből + térkép
poi = S['poi'].copy()
poi.columns = ['POI-sáv', 'Átlagos darabszám']
poi['POI-sáv'] = poi['POI-sáv'].map({'poi_5p_count': '5p (≤375 m)',
                                     'poi_10p_count': '10p (≤750 m)',
                                     'poi_15p_count': '15p (≤1125 m)'})
display(HTML("<b>Átlagos szolgáltatásszám a hálózati sávokban (elemzési minta):</b><br>"
             + poi.to_html(classes='table table-bordered table-striped', index=False)))

fig_poi = px.bar(poi, x='POI-sáv', y='Átlagos darabszám', color='Átlagos darabszám',
                 color_continuous_scale='Greens', text='Átlagos darabszám',
                 title='Átlagos gyalogos szolgáltatás-ellátottság (db)', template=PLOTLY_TEMPLATE, height=380)
fig_poi.update_layout(coloraxis_showscale=False)
fig_poi.show()

# Térkép: 15 perces ellátottság pontonként
pmap = p.dropna(subset=['poi_15p_count', 'nm_ar_huf', 'geokodolt_lat', 'geokodolt_lon'])
if len(pmap):
    fig_poi_map = px.scatter_map(
        pmap, lat='geokodolt_lat', lon='geokodolt_lon', color='poi_15p_count',
        size='nm_ar_huf', hover_name='cim_teljes', map_style=carto_style_url('positron'),
        color_continuous_scale='Viridis',
        title='Hálózaton elérhető szolgáltatások száma 1125 m-en belül (pontonként)')
    fig_poi_map.update_layout(height=500, margin={"r":0,"t":40,"l":0,"b":0})
    fig_poi_map.show()

at15 = float(poi.loc[poi['POI-sáv'] == '15p (≤1125 m)', 'Átlagos darabszám'].iloc[0])
at5 = float(poi.loc[poi['POI-sáv'] == '5p (≤375 m)', 'Átlagos darabszám'].iloc[0])
print(f"Az elemzési mintában átlagosan {at15:.1f} szolgáltatás érhető el 15 perc sétán belül; "
      f"a közvetlen 5 perces körzetben átlagosan {at5:.1f}.")


## 3. Ponttérkép: a fajlagos árak térbeli eloszlása

**MIT látunk?** Minden pont egy eladó lakás az elemzési mintában; a szín a négyzetméterárat, a pontméret az alapterületet jelzi.

**MIÉRT?** A nyers pontfelhő az első ellenőrzés arra, hogy az ár térben strukturált-e (gócok, gradiensek, sávok menti eltérések), mielőtt bármilyen simítást vagy modellt alkalmaznánk.

**HOGYAN?** A térkép tokenmentes webes alaptérképre épül (carto-positron); a színskála az árak 5–95. percentilisére van kalibrálva, hogy néhány szélsőséges hirdetés ne mossa el a különbségeket.

**Döntési szabály:** a ponttérkép diagnosztika; a következő lépésben a pontonkénti zajt IDW-interpolációval (simított felület) csökkentjük.


In [ ]:
# 3. Ponttérkép (px.scatter_map, carto-positron alaptérkép, token nélkül)
def huf_fmt_2(v):
    return f"{v:,.0f} Ft/m²".replace(',', ' ') if pd.notna(v) else "—"

p_map = p.dropna(subset=['nm_ar_huf', 'geokodolt_lat', 'geokodolt_lon']).copy()
fig1 = px.scatter_map(
    p_map,
    lat='geokodolt_lat', lon='geokodolt_lon',
    color='nm_ar_huf', size='alapterulet_nm',
    hover_name='cim_teljes',
    hover_data={'nm_ar_huf': ':.0f', 'ar_millio_ft': ':.1f', 'alapterulet_nm': True,
                'szobaszam_osszes': True, 'varosresz': True},
    color_continuous_scale='Viridis',
    range_color=[p_map['nm_ar_huf'].quantile(0.05), p_map['nm_ar_huf'].quantile(0.95)],
    zoom=12.2, center=get_map_center(), map_style=carto_style_url('positron'),
    title='Lakások térbeli elhelyezkedése és négyzetméterára (carto-positron alaptérkép)')
fig1.update_layout(height=550, margin={"r":0,"t":40,"l":0,"b":0})
fig1.show()

print(f"A ponttérkép {len(p_map)} lakást ábrázol; a színskála a mintabeli árak "
      f"{huf_fmt_2(p_map['nm_ar_huf'].quantile(0.05))} és {huf_fmt_2(p_map['nm_ar_huf'].quantile(0.95))} közötti tartományára kalibrált.")


## 4. Interpolált árfelület és értékdomborzat (IDW)

**MIT mérünk?** Két egymásra épülő ábrázolást: (a) az árak IDW-interpolációja (távolsággal fordítottan súlyozott átlag, kitevő = 2) ~75 méteres rácson, vetített koordinátákon — színezett felület, izovonalak és a pontos hirdetési pontok rárakva; (b) az **értékdomborzat**: ugyanarról a rácsról 2D „domborzati térkép" a lokális csúcsok és völgyek annotációjával, valamint 3D felület — három választható metrikával (fajlagos ár, log ár, reziduum).

**MIÉRT?** Az egyedi pontok árai zajosak; az IDW-felület folytonos árfelszínt rajzol, amelyen a magas- és alacsony árú gócok, illetve gradiensek közvetlenül leolvashatók. A domborzat csúcsai és völgyei a lokális értékgócokat jelölik ki. A **reziduum-mérték** (a fizikai kontrollok utáni maradék) a vasúton túli „maradék értékdomborzat": a vasúttól független értékkülönbségeket mutatja — városrész-prémiumokat, lokális értékhegyeket és -völgyeket —, ezért érdekes.

**HOGYAN?** A vetítés lokális méter-koordinátákat használ (a szélességi fok koszinuszával korrigált hosszúság: x = lon·cos(mid_lat)·111320, y = lat·110540), így a tengelyek méretarányosak. A rácspontok a cache-elt pontfelhőből (`A.price_surface()`: geokodolt_lat, geokodolt_lon, nm_ar_huf, log_nm_ar, resid) interpolálódnak — megjelenítési lépés, NEM új elemzés. A csúcsok/völgyek detektálása a rácson történik: minden cellát a 3×3-as környezetével vetünk össze (maximum/minimum-szűrés), és a top 5 csúcsot/völgyet ▲/▼ markerrel és értékkel jelöljük. A metrikaválasztó kliensoldali JavaScript, a statikus HTML-exportban is működik.

**Döntési szabály:** a színskála a 2–98. percentilisre kalibrált (a reziduumnál szimmetrikusan, ±max abszolút értékre); a reziduum a kanonikus fizikai kontrollok (méret, szobaszám, panel, lift, erkély, állapot, emelet, épületkor) utáni OLS-maradék.


In [ ]:
# 4. IDW-árfelület és értékdomborzat — valódi interaktív térképen (plotly mapbox, Carto)
# A.price_surface() adja a cache-elt pontokat; a notebook csak megjelenít:
# IDW-interpoláció a rácson, a színezett rácspontok a térképre georeferálva jelennek meg.
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

PS = A.price_surface()
df_ar = PS.dropna(subset=['nm_ar_huf', 'geokodolt_lat', 'geokodolt_lon']).copy()
mid_lat = float(df_ar['geokodolt_lat'].mean())
lon_scale = 111320.0 * np.cos(np.radians(mid_lat))
lat_scale = 110540.0
x_proj = df_ar['geokodolt_lon'].values * lon_scale
y_proj = df_ar['geokodolt_lat'].values * lat_scale

STEP = 75.0
PAD = 150.0
gx = np.arange(x_proj.min() - PAD, x_proj.max() + PAD + STEP / 2, STEP)
gy = np.arange(y_proj.min() - PAD, y_proj.max() + PAD + STEP / 2, STEP)
GX, GY = np.meshgrid(gx, gy)
grid_nodes = np.column_stack((GX.ravel(), GY.ravel()))
pts = np.column_stack((x_proj, y_proj))

def _idw(vals):
    z = np.full(len(grid_nodes), np.nan)
    for i in range(0, len(grid_nodes), 512):
        d = grid_nodes[i:i + 512, None, :] - pts[None, :, :]
        d2 = np.einsum('ijk,ijk->ij', d, d)
        w = 1.0 / np.maximum(d2, 1e-9)
        z[i:i + 512] = (w @ vals) / w.sum(axis=1)
    return z

_metrics = [
    ('nm_ar_huf', 'Fajlagos ár (Ft/m²)', 'Viridis', False, df_ar['nm_ar_huf'].values),
    ('log_nm_ar', 'Log fajlagos ár', 'Viridis', False, df_ar['log_nm_ar'].values),
    ('resid', 'Reziduum (fizikai kontrollok után)', 'RdBu_r', True, df_ar['resid'].values),
]
gdf = pd.DataFrame({'geokodolt_lat': GY.ravel() / lat_scale,
                    'geokodolt_lon': GX.ravel() / lon_scale})
for m, cim, scale, sym, vals in _metrics:
    gdf[m] = _idw(vals)
ter_top_csucs = float(np.nanmax(gdf['nm_ar_huf']))

for m, cim, scale, sym, vals in _metrics:
    fig = px.scatter_map(gdf, lat='geokodolt_lat', lon='geokodolt_lon', color=m,
                         color_continuous_scale=scale, opacity=0.45, size_max=3,
                         map_style=carto_style_url('positron'), zoom=12.2, height=520,
                         title=f'Értékdomborzat — {cim} (IDW-interpolált rácspontok a térképen)')
    if sym:
        fig.update_layout(coloraxis=dict(cmid=0))
    _pts = px.scatter_map(df_ar, lat='geokodolt_lat', lon='geokodolt_lon',
                          map_style=carto_style_url('positron'), opacity=0.9)
    fig.add_trace(_pts.data[0])
    fig.data[-1].update(marker=dict(size=6, color='#334155'), name='Hirdetési pontok',
                        customdata=df_ar['nm_ar_huf'].round(0),
                        hovertemplate='Fajlagos ár: %{customdata:,.0f} Ft/m²<extra></extra>')
    fig.show()

print(f"Értékdomborzat: {len(df_ar)} pontos hirdetés IDW-interpolálva ({len(grid_nodes):,} rácspont, "
      f"~{STEP:.0f} m rács), {len(_metrics)} mutató (ár / log ár / reziduum) valódi interaktív "
      f"térképen (Carto pozitron, georeferált).")

## 5. Hálózati távolságok és a kerülőfaktor

**MIT mérünk?** Azt, hogy a valódi gyalogos útvonal hossza mennyivel haladja meg a légvonali távolságot. A **kerülőfaktor** a hálózati (utcahálózaton mért legrövidebb) út és a légvonali távolság hányadosa.

**MIÉRT?** A légvonali távolság szisztematikusan alulbecsüli a valós gyaloglást: az utcahálózat kerülői, a kényszerű átkelők és a zsákutcák miatt a tényleges sétaút jellemzően másfél-kétszeresére nyúlik. Ezért minden elérhetőségi elemzés hálózati távolságokra épül, és a kerülőfaktor a feldolgozó lánc állandó ellenőrző mutatója.

**HOGYAN?** A hálózati távolságok az utcahálózat-gráfon futó legrövidebb út kereséséből származnak (a feldolgozó láncban, gyorsítótárban); a notebook a medián kerülőfaktort és a hálózati távolságok mediánjait mutatja meg hálózatonként.

**Döntési szabály:** ha a kerülőfaktor szélsőséges (egynél kisebb vagy nagyon nagy), az adathibára utal; a jellemző 1,5–2 közötti érték az egészséges városi úthálózat jele.


In [ ]:
# 5. Kerülőfaktor és hálózati távolságok — számítás a cache adataiból, f-string kimenet
detour_rows = p.dropna(subset=['tavolsag_vasut_halozati_m', 'tavolsag_vasut_m'])
detour = (detour_rows['tavolsag_vasut_halozati_m'] / detour_rows['tavolsag_vasut_m'].replace(0, np.nan)).dropna()
detour_med = float(detour.median())

iso2 = S['izokronok']
net_lines = []
for _, r in iso2.iterrows():
    net_lines.append(f"{r['halozat']}: 15 perces körzetben {int(r['15p_N'])} lakás, "
                     f"folytonos távolság mediánja {r['median_tav_m']:.0f} m "
                     f"({int(r['folytonos_kitoltott'])} kitöltött érték)")
print("Hálózati távolságok hálózatonként: " + "; ".join(net_lines))
print(f"A kerülőfaktor (hálózati út / légvonali távolság) mediánja az elemzési mintán: "
      f"{detour_med:.2f} — azaz a valós sétaút mediánban közel {detour_med:.2f}-szerese "
      f"a mértani távolságnak (n = {len(detour)} megfigyelés).")


## Összefoglalás

A fejezet térbeli méréseit egyetlen mondatfolyamba rendezzük: minden szám a közös elemző modul gyorsítótárából, futásidőben illesztve.


In [ ]:
# Záró összefoglaló — minden szám a cache-ből, f-stringgel (nincs hardkódolt érték)
def _n2(v):
    return f"{int(v):,}".replace(',', ' ')

iso3 = S['izokronok']; sav3 = S['immisszios_savok']; poi3 = S['poi']
poi15 = float(poi3.loc[poi3['poi'] == 'poi_15p_count', 'atlag'].iloc[0])
vas = iso3.loc[iso3['halozat'] == 'vasut']
kp = iso3.loc[iso3['halozat'] == 'kotottpalya']

print(
    f"Az elemzési minta {_n2(len(p))} lakásából {_n2(int(vas['15p_N'].iloc[0]))} lakás éri el a legközelebbi vasútállomást "
    f"15 perc sétán belül, a kötöttpályás hálózat valamely megállóját pedig {_n2(int(kp['15p_N'].iloc[0]))} lakás. "
    f"A közvetlen immissziós sávban (<150 m a vágánytól) {_n2(int(sav3['N'].iloc[0]))} lakás fekszik. "
    f"A gyalogos körzetben átlagosan {poi15:.1f} szolgáltatás érhető el 15 percen belül. "
    f"A hálózati sétaút a légvonalihoz képest mediánban {detour_med:.2f}-szeres (kerülőfaktor). "
    f"A térbeli árszerkezet az IDW-rácson {_n2(len(grid_nodes))} rácspontból épül fel, "
    f"a domborzat legmagasabb lokális árcsúcsa {fmt_huf(ter_top_csucs) if pd.notna(ter_top_csucs) else '—'}."
)
